# M10 · Sistemas multi-agente: prática

1. Mecânica do LangGraph sem LLM: estado, reducers, fan-out paralelo, checkpoints e `interrupt`.
2. Um supervisor que escolhe o especialista seguinte.
3. A equipa de pesquisa de mercado, com aprovação humana.
4. Retomar e voltar atrás no tempo com checkpoints.
5. Avaliar pela trajetória e pelo resultado.

**Requisitos:** `uv sync --extra agentes`, `ANTHROPIC_API_KEY` (secções 2 a 5). A secção 1 não usa API.

Lição de apoio: [licao.html](licao.html)

In [ ]:
import json, operator, os, sys, time
from pathlib import Path
from typing import Annotated, TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send, interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()
UC = Path("use-case").resolve()

## 1. Mecânica do LangGraph sem LLM

Três «investigadores» falsos correm em paralelo (cada um demora 1 segundo). O reducer `operator.add` junta os resultados numa lista. Depois o grafo para à espera de aprovação.

In [ ]:
class E(TypedDict, total=False):
    temas: list[str]
    achados: Annotated[list[str], operator.add]
    decisao: str
    final: str

def planear(s):
    return {"temas": ["demografia", "concorrência", "rendas"]}

def investigar(entrada):            # recebe o payload do Send, não o estado
    time.sleep(1)
    return {"achados": [f"achado sobre {entrada['tema']}"]}

def aprovar(s):
    return {"decisao": interrupt({"para_aprovar": s["achados"]})}

def escrever(s):
    return {"final": f"Relatório com {len(s['achados'])} achados. Nota humana: {s['decisao']}"}

g = StateGraph(E)
for nome, f in [("planear", planear), ("investigar", investigar), ("aprovar", aprovar), ("escrever", escrever)]:
    g.add_node(nome, f)
g.add_edge(START, "planear")
g.add_conditional_edges("planear", lambda s: [Send("investigar", {"tema": t}) for t in s["temas"]], ["investigar"])
g.add_edge("investigar", "aprovar")
g.add_edge("aprovar", "escrever")
g.add_edge("escrever", END)
mini = g.compile(checkpointer=InMemorySaver())
print(mini.get_graph().draw_mermaid())

In [ ]:
cfg = {"configurable": {"thread_id": "mini-1"}}
t0 = time.time()
r = mini.invoke({}, cfg)
print(f"{time.time() - t0:.1f} s (3 investigadores de 1 s em paralelo)")
print("Interrompido em:", mini.get_state(cfg).next, "\nPedido:", r["__interrupt__"][0].value)

In [ ]:
r = mini.invoke(Command(resume="aprovado, mas confirmar rendas"), cfg)
print(r["final"])

**Exercício 1.1.** Troca `operator.add` por nada (`achados: list[str]`). O que acontece quando três nós escrevem a mesma chave no mesmo passo? Porque precisa o fan-out de um reducer?

## 2. Um supervisor

O supervisor decide, a cada passo, quem age a seguir. Aqui os especialistas são simples chamadas ao modelo; num sistema real seriam agentes com ferramentas.

In [ ]:
from typing import Literal
from anthropic import Anthropic
from pydantic import BaseModel, Field

llm = Anthropic()
MODELO = os.getenv("MODELO", "claude-haiku-5-5")
MODELO_FORTE = os.getenv("MODELO_FORTE", "claude-sonnet-5-5")

class Decisao(BaseModel):
    proximo: Literal["analista", "redator", "revisor", "FIM"]
    instrucao: str = Field(description="O que o especialista deve fazer agora, de forma autónoma")

ESPECIALISTAS = {
    "analista": "Calculas números a partir dos dados fornecidos. Mostra as contas.",
    "redator": "Escreves textos curtos e claros para a direção, em português de Portugal.",
    "revisor": "Verificas se o texto está correto face aos dados e apontas erros concretos.",
}

class S(TypedDict, total=False):
    pedido: str
    historico: Annotated[list[str], operator.add]
    decisao: dict

def supervisor(s):
    r = llm.messages.parse(model=MODELO_FORTE, max_tokens=2000, output_format=Decisao, messages=[{"role": "user", "content":
        f"Coordenas especialistas: {list(ESPECIALISTAS)}. Pedido: {s['pedido']}\n\nTrabalho feito até agora:\n" +
        "\n\n".join(s.get("historico", [])) + "\n\nEscolhe quem age a seguir, ou FIM se o pedido está cumprido e revisto."}])
    d = r.parsed_output
    print(f"🧭 supervisor → {d.proximo}: {d.instrucao[:100]}")
    return {"decisao": d.model_dump()}

def especialista(s):
    nome, instr = s["decisao"]["proximo"], s["decisao"]["instrucao"]
    r = llm.messages.create(model=MODELO, max_tokens=1500, system=ESPECIALISTAS[nome], messages=[{"role": "user", "content":
        f"{instr}\n\nContexto:\n{s['pedido']}\n\n" + "\n\n".join(s.get("historico", []))}])
    return {"historico": [f"[{nome}] " + r.content[-1].text]}

g = StateGraph(S)
g.add_node("supervisor", supervisor)
g.add_node("especialista", especialista)
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", lambda s: END if s["decisao"]["proximo"] == "FIM" else "especialista")
g.add_edge("especialista", "supervisor")
sup = g.compile()

out = sup.invoke({"pedido": "Vendas de novembro: Online 182 000 €, Porto 61 000 €, Lisboa 64 000 €, Braga 30 000 €, Faro 21 000 €. "
                            "Em novembro de 2024 o total foi 318 000 €. Escreve um parágrafo para a direção com a variação."},
                 {"recursion_limit": 12})
print("\n", out["historico"][-1])

**Exercício 2.1.** Conta quantas chamadas ao modelo foram precisas. Resolve o mesmo pedido com uma só chamada a `MODELO_FORTE`. Para esta tarefa, o supervisor compensa?

## 3. A equipa de pesquisa de mercado

O `equipa.py` junta tudo: planeador, investigadores web e SQL em paralelo, revisor, aprovação humana e redator. Usamos um checkpointer em memória; o script de linha de comandos usa SQLite.

In [ ]:
sys.path.insert(0, str(UC))
import equipa

app = equipa.construir(InMemorySaver())
cfg = {"configurable": {"thread_id": "coimbra-1"}, "recursion_limit": 40}
PERGUNTA = "Vale a pena a Loja Lusa abrir uma loja física em Coimbra em 2026?"

for evento in app.stream({"pergunta": PERGUNTA}, cfg, stream_mode="updates"):
    for no, upd in evento.items():
        if no == "planear":
            for t in upd["tarefas"]:
                print(f"📋 {t['id']} [{t['fonte']}] {t['pergunta']}")
        elif no == "investigar":
            for i, a in upd["achados"].items():
                print(f"🔎 {i}: {len(a['fontes'])} fontes")
        elif no == "rever":
            print(f"🧐 revisão: aprovado={upd['revisao']['aprovado']}, problemas={len(upd['revisao']['problemas'])}")
        elif no == "__interrupt__":
            print("⏸️  à espera de aprovação")

In [ ]:
estado = app.get_state(cfg)
equipa.mostrar_achados(estado.tasks[0].interrupts[0].value)

In [ ]:
decisao = input("'ok' ou instruções para o redator: ") or "ok"
final = app.invoke(Command(resume=decisao), cfg)
print(final["relatorio"])
print(json.dumps(equipa.custo(final["uso"]), indent=1))

**Exercício 3.1.** Repara no custo por nó (`final["uso"]`). Que nó gasta mais? Experimenta `MODELO_FORTE=claude-haiku-5-5` só para o planeador e compara a qualidade do plano.

## 4. Checkpoints: histórico e viagem no tempo

Cada passo ficou gravado. Podemos ver o histórico e voltar a um ponto anterior, por exemplo para reescrever o relatório com outra instrução sem repetir as pesquisas.

In [ ]:
historico = list(app.get_state_history(cfg))
for h in historico:
    print(h.config["configurable"]["checkpoint_id"][-8:], "próximo:", h.next, "| chaves:", sorted(h.values)[:6])

In [ ]:
# Criamos um ramo a partir do checkpoint anterior à aprovação, como se a pessoa tivesse dado outra instrução
antes_de_aprovar = next(h for h in historico if h.next == ("aprovar",))
ramo = app.update_state(antes_de_aprovar.config, {"decisao_humana": "Escreve só a recomendação, em 5 linhas, e sê conservador."}, as_node="aprovar")
outra = app.invoke(None, ramo)      # corre só o que falta: escrever
print(outra["relatorio"][:1500])

Repara que não houve novas pesquisas: `update_state(..., as_node="aprovar")` grava um novo checkpoint como se o nó de aprovação tivesse devolvido outra decisão, e o grafo continua daí. O histórico original fica intacto. Com o `SqliteSaver` do script (`equipa.py --retomar <thread_id>`) isto funciona também depois de fechares o terminal.

**Exercício 4.1.** Corre `uv run python equipa.py "..."` no terminal, interrompe com Ctrl+C durante a investigação e retoma com `--retomar`. Que nós voltaram a correr?

## 5. Avaliar pela trajetória e pelo resultado

O `avaliar.py` corre cada pergunta k vezes com aprovação automática e verifica a trajetória (ordem dos nós, fontes exigidas no plano, esforço proporcional, orçamento) e o resultado (achados com fonte, citações válidas e, opcionalmente, um juiz).

In [ ]:
import avaliar

caso = {"id": "factual", "pergunta": "Qual foi a loja da Loja Lusa com mais vendas em 2025?", "exige": ["interno"], "max_tarefas": 3, "max_usd": 0.15}
resultados = []
for i in range(2):
    traj, est = avaliar.executar(caso["pergunta"], i)
    checks = avaliar.verificar(caso, traj, est)
    resultados.append(all(checks.values()))
    print(f"#{i + 1} trajetória: {' → '.join(traj)}")
    print("   ", {k: "✅" if v else "❌" for k, v in checks.items()})
print(f"taxa de sucesso {sum(resultados) / len(resultados):.0%}, pass^2: {all(resultados)}")

Uma pergunta factual sobre dados internos devia dar um plano curto, só com tarefas internas. Se o planeador lançar pesquisas web desnecessárias, o `esforco_proporcional` ou o orçamento falham: é uma falha de trajetória, mesmo que a resposta esteja certa.

**Exercício 5.1.** Corre `uv run python avaliar.py --k 3 --juiz` no terminal. Em que pergunta o pass^3 falha? Melhora o `SYSTEM_PLANEADOR` e volta a medir.

**Exercício 5.2 (comparação honesta).** Constrói um só agente com as mesmas ferramentas (pesquisa web e SQL) e um prompt que pede o mesmo relatório. Compara custo, tempo e a nota do juiz nas 4 perguntas. Em que perguntas a equipa compensa?

## Use case: equipa de pesquisa de mercado

```bash
cd modulos/10-sistemas-multi-agente/use-case
uv run python equipa.py "A categoria Cozinha está a crescer mais na Loja Lusa do que no mercado?"
uv run python avaliar.py --k 2
```

**Desafio:** acrescenta um investigador de concorrência que usa o servidor de ficheiros MCP do módulo 9 para ler relatórios internos em PDF, e uma regra no revisor que exige pelo menos uma fonte oficial (INE, Pordata, Banco de Portugal) em perguntas de mercado.